# Climate Mitigation Potential widget data preparation

## Load libraries

In [1]:
import pandas as pd
import json
import geopandas as gpd
import subprocess


## Load and clean data
Mitigation potential data is stored in a CSV on the [GCS folder](https://console.cloud.google.com/storage/browser/wetlands-gap-map/original_data;tab=objects?inv=1&invt=Ab12FA&prefix=&forceOnObjectsSortingFiltering=false).  
It contains all countries, needs to be filtered to Sahel countries.  


In [2]:
df = pd.read_csv('https://storage.googleapis.com/wetlands-gap-map/original_data/Mitigation_Country_clean.csv')
sel_cols = ['ISO',
        'Reduce deforestation AVG', 'Reforestation AVG', 'Forest management AVG',
       'Grassland and savanna fire mgmt ',
       'Reduce peatland degradation', 'Peatland restoration',
       'Reduce Mangrove conversion', 'Mangrove restoration'] # Selected columns need to be reviewed for final data
df = df[sel_cols]
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower().str.replace('_avg', '')
df.head()

,iso,reduce_deforestation,reforestation,forest_management,grassland_and_savanna_fire_mgmt,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration
0,AFG,54.7488,84.1559,100.6039,NaN,NaN,NaN,NaN,NaN
1,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ALB,NaN,54.0851,9.6573,NaN,NaN,NaN,NaN,NaN
3,DZA,NaN,126.3656,6.4040,NaN,NaN,NaN,NaN,NaN
4,ASM,NaN,547.1500,0.0000,NaN,NaN,NaN,NaN,NaN


In [3]:
#Filter to Sahel countries
countries = pd.read_csv('https://storage.googleapis.com/wetlands-gap-map/original_data/countries_sahel.csv')
print(f'Initial countries: {len(df)}')
df = df[df['iso'].isin(countries['code'])]
print(f'Countries after filtering: {len(df)}')

Initial countries: 250
Countries after filtering: 22


## Process data.  
Process and reshape the data for match needed format.

In [11]:
# Pivot to long table and create location id column
df_long = df.melt(id_vars=['iso'], var_name='intervention', value_name='value')
df_long['location'] = df_long['iso'].apply(lambda x: f'adminregion_{x.lower()}')
df_long = df_long[df_long['value'].notna()]
df_long = df_long[['location', 'intervention', 'value']]
df_long

,location,intervention,value
0,adminregion_ben,reduce_deforestation,262.9976
1,adminregion_bfa,reduce_deforestation,115.6229
2,adminregion_cmr,reduce_deforestation,374.3571
3,adminregion_caf,reduce_deforestation,262.3109
4,adminregion_tcd,reduce_deforestation,225.7684
...,...,...,...
165,adminregion_gnb,mangrove_restoration,704.0000
166,adminregion_ken,mangrove_restoration,704.0000
170,adminregion_nga,mangrove_restoration,704.0000
171,adminregion_sen,mangrove_restoration,704.0000


In [12]:
df_long['intervention'].unique().tolist()

['reduce_deforestation',
 'reforestation',
 'forest_management',
 'grassland_and_savanna_fire_mgmt',
 'reduce_peatland_degradation',
 'peatland_restoration',
 'reduce_mangrove_conversion',
 'mangrove_restoration']

In [13]:
intervention_type_map = {'reduce_deforestation':'protection',
 'reforestation':'restoration',
 'forest_management':'protection',
 'grassland_and_savanna_fire_mgmt':'protection',
 'reduce_peatland_degradation':'protection',
 'peatland_restoration':'restoration',
 'reduce_mangrove_conversion':'protection',
 'mangrove_restoration':'restoration'}
ecosystem_type_map = {'reduce_deforestation':'non-wetlands',
 'reforestation':'non-wetlands',
 'forest_management':'non-wetlands',
 'grassland_and_savanna_fire_mgmt':'non-wetlands',
 'reduce_peatland_degradation':'wetlands',
 'peatland_restoration':'wetlands',
 'reduce_mangrove_conversion':'wetlands',
 'mangrove_restoration':'wetlands'}
color_map = {'reduce_deforestation':'#628c5f',
 'reforestation':'#98ed93',
 'forest_management':'#255c21',
 'grassland_and_savanna_fire_mgmt':'#10ba04',
 'reduce_peatland_degradation':'#138d91',
 'peatland_restoration':'#c690f5',
 'reduce_mangrove_conversion':'#71419c',
 'mangrove_restoration':'#51dce0'}

In [14]:
df_long_complete = df_long.copy()
df_long_complete['type'] = df_long_complete['intervention'].map(intervention_type_map)
df_long_complete['group'] = df_long_complete['intervention'].map(ecosystem_type_map)
df_long_complete['color'] = df_long_complete['intervention'].map(color_map)
df_long_complete = df_long_complete[['location', 'intervention', 'color', 'type', 'group', 'value']]

df_long_complete.head()

,location,intervention,color,type,group,value
0,adminregion_ben,reduce_deforestation,#628c5f,protection,non-wetlands,262.9976
1,adminregion_bfa,reduce_deforestation,#628c5f,protection,non-wetlands,115.6229
2,adminregion_cmr,reduce_deforestation,#628c5f,protection,non-wetlands,374.3571
3,adminregion_caf,reduce_deforestation,#628c5f,protection,non-wetlands,262.3109
4,adminregion_tcd,reduce_deforestation,#628c5f,protection,non-wetlands,225.7684


In [15]:
# Combine data for each row as dictionary
df_location = df_long_complete.copy()
df_location['data'] = df_location.apply(lambda x: {'id': 'mitigation-' + str(x.name),
                                                 'label': x['intervention'],
                                                 'type': x['type'],
                                                 'group': x['group'],
                                                 'value': x['value'],
                                                 'format': 'number',
                                                 'unit': 'tCO2e/ha',
                                                 'color': x['color']}, axis=1)
df_location.head()

,location,intervention,color,type,group,value,data
0,adminregion_ben,reduce_deforestation,#628c5f,protection,non-wetlands,262.9976,"{'id': 'mitigation-0', 'label': 'reduce_defore..."
1,adminregion_bfa,reduce_deforestation,#628c5f,protection,non-wetlands,115.6229,"{'id': 'mitigation-1', 'label': 'reduce_defore..."
2,adminregion_cmr,reduce_deforestation,#628c5f,protection,non-wetlands,374.3571,"{'id': 'mitigation-2', 'label': 'reduce_defore..."
3,adminregion_caf,reduce_deforestation,#628c5f,protection,non-wetlands,262.3109,"{'id': 'mitigation-3', 'label': 'reduce_defore..."
4,adminregion_tcd,reduce_deforestation,#628c5f,protection,non-wetlands,225.7684,"{'id': 'mitigation-4', 'label': 'reduce_defore..."


In [16]:
# Combine data for each location as array of dictionaries from the data column

df_location = df_location.groupby('location')['data'].apply(list).reset_index()
df_location['id'] = 'mitigation-potential-' + df_location.index.astype(str)
#df_location['data'] = df_location['data'].apply(json.dumps)
df_location = df_location[['id', 'location', 'data']]
df_location.head(10)

,id,location,data
0,mitigation-potential-0,adminregion_ben,"[{'id': 'mitigation-0', 'label': 'reduce_defor..."
1,mitigation-potential-1,adminregion_bfa,"[{'id': 'mitigation-1', 'label': 'reduce_defor..."
2,mitigation-potential-2,adminregion_caf,"[{'id': 'mitigation-3', 'label': 'reduce_defor..."
3,mitigation-potential-3,adminregion_civ,"[{'id': 'mitigation-5', 'label': 'reduce_defor..."
4,mitigation-potential-4,adminregion_cmr,"[{'id': 'mitigation-2', 'label': 'reduce_defor..."
5,mitigation-potential-5,adminregion_eri,"[{'id': 'mitigation-28', 'label': 'reforestati..."
6,mitigation-potential-6,adminregion_eth,"[{'id': 'mitigation-7', 'label': 'reduce_defor..."
7,mitigation-potential-7,adminregion_gha,"[{'id': 'mitigation-9', 'label': 'reduce_defor..."
8,mitigation-potential-8,adminregion_gin,"[{'id': 'mitigation-10', 'label': 'reduce_defo..."
9,mitigation-potential-9,adminregion_gmb,"[{'id': 'mitigation-8', 'label': 'reduce_defor..."


In [19]:
translations_dict = {'en': {
    'labels': {
        'reduce_deforestation': 'Reduce deforestation',
        'reforestation': 'Reforestation',
        'forest_management': 'Forest management',
        'grassland_and_savanna_fire_mgmt': 'Grassland and savanna fire management',
        'reduce_peatland_degradation': 'Reduce peatland degradation',
        'peatland_restoration': 'Peatland restoration',
        'reduce_mangrove_conversion': 'Reduce mangrove conversion',
        'mangrove_restoration': 'Mangrove restoration'
    }
    }
}

# add full dictionary on each row of the field 'locale'
df_location['locale'] = df_location.apply(lambda x: translations_dict, axis=1)
df_location['indicator'] = 'wetlands-mitigation-potential'
df_location = df_location[['id', 'location', 'indicator', 'data', 'locale']]
df_location.sample(10)

,id,location,indicator,data,locale
10,mitigation-potential-10,adminregion_gnb,wetlands-mitigation-potential,"[{'id': 'mitigation-11', 'label': 'reduce_defo...",{'en': {'labels': {'reduce_deforestation': 'Re...
3,mitigation-potential-3,adminregion_civ,wetlands-mitigation-potential,"[{'id': 'mitigation-5', 'label': 'reduce_defor...",{'en': {'labels': {'reduce_deforestation': 'Re...
8,mitigation-potential-8,adminregion_gin,wetlands-mitigation-potential,"[{'id': 'mitigation-10', 'label': 'reduce_defo...",{'en': {'labels': {'reduce_deforestation': 'Re...
9,mitigation-potential-9,adminregion_gmb,wetlands-mitigation-potential,"[{'id': 'mitigation-8', 'label': 'reduce_defor...",{'en': {'labels': {'reduce_deforestation': 'Re...
13,mitigation-potential-13,adminregion_mrt,wetlands-mitigation-potential,"[{'id': 'mitigation-14', 'label': 'reduce_defo...",{'en': {'labels': {'reduce_deforestation': 'Re...
20,mitigation-potential-20,adminregion_tgo,wetlands-mitigation-potential,"[{'id': 'mitigation-20', 'label': 'reduce_defo...",{'en': {'labels': {'reduce_deforestation': 'Re...
11,mitigation-potential-11,adminregion_ken,wetlands-mitigation-potential,"[{'id': 'mitigation-12', 'label': 'reduce_defo...",{'en': {'labels': {'reduce_deforestation': 'Re...
15,mitigation-potential-15,adminregion_nga,wetlands-mitigation-potential,"[{'id': 'mitigation-16', 'label': 'reduce_defo...",{'en': {'labels': {'reduce_deforestation': 'Re...
2,mitigation-potential-2,adminregion_caf,wetlands-mitigation-potential,"[{'id': 'mitigation-3', 'label': 'reduce_defor...",{'en': {'labels': {'reduce_deforestation': 'Re...
5,mitigation-potential-5,adminregion_eri,wetlands-mitigation-potential,"[{'id': 'mitigation-28', 'label': 'reforestati...",{'en': {'labels': {'reduce_deforestation': 'Re...


### Save data to JSON file

In [ ]:
#df_location.to_csv('../data/processed/test_mitigation_indicator_data.csv', index=False)

In [22]:
df_location.to_json('../data/processed/test_mitigation_indicator_data.json', orient='records', lines=True)

## Vector layer

In [2]:
countries_poly = gpd.read_file('../data/processed/countries_sahel.geojson')
countries_poly.head(3)

,name,bbox,ISO3,geometry
0,Benin,"{'bbox': [0.776667, 6.0398696, 3.8451454, 12.4...",BEN,"POLYGON ((0.77667 10.37667, 0.79220 10.36589, ..."
1,Burkina Faso,"{'bbox': [-5.513207, 9.4104718, 2.4089717, 15....",BFA,"POLYGON ((-5.51321 10.43079, -5.51319 10.43066..."
2,Cameroon,"{'bbox': [8.3822176, 1.6517945, 16.1911011, 13...",CMR,"POLYGON ((8.38222 4.34753, 8.39136 4.34017, 8...."


In [9]:
mitigation_wetlands_df = pd.read_csv('https://storage.googleapis.com/wetlands-gap-map/original_data/Mitigation_Country_clean.csv')
sel_cols = ['ISO',
        'Reduce deforestation AVG', 'Reforestation AVG', 'Forest management AVG',
       'Grassland and savanna fire mgmt ',
       'Reduce peatland degradation', 'Peatland restoration',
       'Reduce Mangrove conversion', 'Mangrove restoration'] # Selected columns need to be reviewed for final data
mitigation_wetlands_df = mitigation_wetlands_df[sel_cols]
mitigation_wetlands_df.columns = mitigation_wetlands_df.columns.str.strip().str.replace(' ', '_').str.lower().str.replace('_avg', '')
mitigation_wetlands_df.sample(5)


,iso,reduce_deforestation,reforestation,forest_management,grassland_and_savanna_fire_mgmt,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration
35,BFA,115.6229,199.2904,31.2582,16.8085,NaN,870.0000,NaN,NaN
138,MHL,NaN,547.1500,48.3668,NaN,NaN,NaN,NaN,NaN
218,TWN,NaN,NaN,NaN,NaN,NaN,NaN,1416.6444,704.0
75,FRA,NaN,23.5123,1.5015,NaN,360.9756,758.3871,NaN,NaN
121,LAO,313.7757,169.0422,0.3652,NaN,NaN,1734.8077,NaN,NaN


In [10]:
mitigation_wetlands_df['wetlands_mitigation_AVG'] = mitigation_wetlands_df[['reduce_peatland_degradation',
                                                                            'peatland_restoration',
                                                                            'reduce_mangrove_conversion',
                                                                            'mangrove_restoration']].mean(axis=1)
mitigation_wetlands_df['other_mitigation_AVG'] = mitigation_wetlands_df[['reduce_deforestation',
                                                                          'reforestation',
                                                                          'forest_management',
                                                                          'grassland_and_savanna_fire_mgmt']].mean(axis=1)
mitigation_wetlands_df.head(3)


,iso,reduce_deforestation,reforestation,forest_management,grassland_and_savanna_fire_mgmt,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration,wetlands_mitigation_AVG,other_mitigation_AVG
0,AFG,54.7488,84.1559,100.6039,NaN,NaN,NaN,NaN,NaN,NaN,79.8362
1,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ALB,NaN,54.0851,9.6573,NaN,NaN,NaN,NaN,NaN,NaN,31.8712


In [12]:
mitigation_gdf = mitigation_wetlands_df.merge(countries_poly, left_on='iso', right_on='ISO3')
mitigation_gdf = gpd.GeoDataFrame(mitigation_gdf, geometry='geometry', crs='EPSG:4326')
mitigation_gdf.drop(columns=['bbox', 'ISO3'], inplace=True)
mitigation_gdf.head(3)

,iso,reduce_deforestation,reforestation,forest_management,grassland_and_savanna_fire_mgmt,reduce_peatland_degradation,peatland_restoration,reduce_mangrove_conversion,mangrove_restoration,wetlands_mitigation_AVG,other_mitigation_AVG,name,geometry
0,BEN,262.9976,204.0629,3.5320,52.9058,2565.0000,3180.0000,847.0226,704.0,1824.00565,130.874575,Benin,"POLYGON ((0.77667 10.37667, 0.7922 10.36589, 0..."
1,BFA,115.6229,199.2904,31.2582,16.8085,NaN,870.0000,NaN,NaN,870.00000,90.745000,Burkina Faso,"POLYGON ((-5.51321 10.43079, -5.51319 10.43066..."
2,CMR,374.3571,218.1430,0.6945,20.0409,1045.2857,1978.6957,1470.9252,704.0,1299.72665,153.308875,Cameroon,"POLYGON ((8.38222 4.34753, 8.39136 4.34017, 8...."


In [13]:
mitigation_gdf.to_file('../data/processed/mitigation_sahel.geojson', driver='GeoJSON')

In [ ]:
#Create mbtiles
infile = '../data/processed/mitigation_sahel.geojson'
outfile = '../data/processed/mitigation_sahel.mbtiles'
subprocess.run(['tippecanoe', '-zg', '-f', '-o', outfile, infile])